In [ ]:
import kagglehub
import sklearn as sk
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd

df = pd.read_csv(path + "/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=10, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=["Order_ID"],inplace = True)

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

In [ ]:
df['Weather'].fillna(df["Weather"].mode()[0],inplace=True)
df['Traffic_Level'].fillna(df["Traffic_Level"].mode()[0],inplace=True)
df['Time_of_Day'].fillna(df["Time_of_Day"].mode()[0],inplace=True)
df['Courier_Experience_yrs'].fillna(df["Courier_Experience_yrs"].mean(),inplace=True)
df['Delivery_Time'].fillna(df["Delivery_Time"].mean(),inplace=True)
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

catgorical = df.select_dtypes(include= "object")


for col in catgorical:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
df.head()

In [ ]:
# Task 5: Write your code here:
features = df.columns.drop("Delivery_Time")

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df[df.columns.drop("Delivery_Time")]
Y = df["Delivery_Time"]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:


kf = KFold(n_splits=5, shuffle=True, random_state=42)
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)


mae_scores = []


for train_idx, val_idx in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))


mae_scores = np.array(mae_scores)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")

In [ ]:
print(f"train : {max(mae_scores):.2f}")
print(f'test : {mae:.2f}')

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=10, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: